In [17]:
import os
import operator
import asyncio

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig

from typing_extensions import TypedDict
from typing import List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [29]:
load_dotenv()

MODEL=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

fanout_worker_llm=ChatGroq(
    model=MODEL
)

editor_base_llm=ChatGroq(
    model=MODEL
)

In [30]:
class OverallState(TypedDict):
    topic: str
    subtopics: List[str]
    research_results: Annotated[List[str], operator.add] # This will be coming from our fanout nodes
    final_report: str


class ResearchState(TypedDict): # Private State
    research_topic: str

In [31]:
class SubtopicPlan(BaseModel):
    subtopics: List[str]=Field(..., description="List of subtopic to research")

In [ ]:
async def generate_subtopics(state: OverallState):
    topic=state["topic"]

    prompt=ChatPromptTemplate.from_messages([
        ("system", "Break the given research topic into clear, specific subtopics for a comprehensive report."),
        ("human", "Topic: {topic}")
    ])

    planner_chain=prompt | fanout_worker_llm.with_structured_output(SubtopicPlan)

    plan: SubtopicPlan=await planner_chain.ainvoke({
        "topic": topic,
    })

    return {
        "subtopics": plan.subtopics
    }

In [33]:
async def research_subtopic(state: ResearchState):
    subtopic=state["research_topic"]

    prompt=ChatPromptTemplate.from_messages([
        ("system", "Provide a concise, insightful 1-paragraph research summary of the following subtopic with key facts and insights."),
        ("human", "Subtopic: {subtopic}")
    ])

    research_chain=prompt | fanout_worker_llm

    result=await research_chain.ainvoke({
        "subtopic": subtopic,
    })

    finding=f"### {subtopic} ###\n{result.content}"

    return {
        "research_results": [finding]
    }

In [34]:
async def compile_report(state: OverallState):
    combined_notes="\n\n".join(state["research_results"])

    prompt=ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a senior research editor. Synthesize the following subtopic research notes "
            "into a well-structured, cohesive executive report with an introduction and conclusion."
        ),
        ("human", "Main Topic: {topic}\nSubtopic Research Notes: {combined_notes}")
    ])

    editor_chain=prompt | editor_base_llm

    response=await editor_chain.ainvoke({
        "topic": state["topic"],
        "combined_notes": combined_notes
    })

    return {
        "final_report": response.content
    }

In [35]:
def map_to_research(state: OverallState):
    return [
        Send(
            node="research_subtopic",
            arg={"research_topic": s}
        )
        for s in state["subtopics"]
    ]

In [36]:
builder=StateGraph(OverallState)

builder.add_node("generate_subtopics", generate_subtopics)
builder.add_node("research_subtopic", research_subtopic)
builder.add_node("compile_report", compile_report)

builder.add_edge(START, "generate_subtopics")
builder.add_conditional_edges("generate_subtopics", map_to_research)
builder.add_edge("research_subtopic", "compile_report")
builder.add_edge("compile_report", END)

graph=builder.compile()

In [37]:
initial_state={
    "topic": "The evolution of GPU in the last 10 years",
    "subtopics": [],
    "research_results": [],
    "final_report": ""
}

In [38]:
result=await graph.ainvoke(initial_state)

In [39]:
print(result["final_report"])

**Executive Report**  
**The Evolution of GPUs Over the Past Decade**  
*Prepared for: Senior Management & Strategic Planning*  
*Date: 15 Nov 2026*  

---

### 1. Executive Summary  

Over the last ten years, GPUs have transformed from a niche graphics accelerator into the backbone of modern high‑performance computing, AI, and immersive media. Key drivers of this evolution include:

| Driver | Impact | Milestone |
|--------|--------|-----------|
| **Performance‑per‑Watt** | 3–5× gains (Pascal → Ada Lovelace) | CUDA core density, RT & Tensor cores |
| **Specialized Cores** | Dedicated AI (Tensor), Ray‑tracing (RT) | Volta → Turing → Ampere → Ada |
| **Memory Bandwidth** | 3–4× increase (GDDR6X → HBM2E) | Higher throughput for ML & 8K rendering |
| **API & Shader Modernization** | Cross‑platform portability, real‑time ray‑tracing | Vulkan, DXR, Metal, SPIR‑V |
| **Data‑Center Adoption** | GPU‑as‑a‑Service, elastic scaling | NVIDIA A100, AMD MI250, Intel Xe‑HPG |
| **Market Fragmentation

In [40]:
print(result["subtopics"])

['Historical milestones and timeline of GPU development over the last decade', 'Key architectural innovations (e.g., CUDA cores, tensor cores, ray tracing cores)', 'Evolution of GPU memory technologies (GDDR5 to GDDR6/6X, HBM, HBM2, HBM2E)', 'Advancements in shading languages and graphics APIs (OpenGL, DirectX, Vulkan, Metal)', 'Impact of GPU on AI/ML workloads and the rise of tensor cores', 'Energy efficiency improvements and power consumption trends', 'GPU integration in data centers and cloud services', 'Changes in market dynamics: major players, market share, and pricing strategies', 'Emerging use cases beyond gaming (e.g., scientific computing, autonomous vehicles, virtual reality)', 'Future outlook: anticipated architectural trends and challenges']


In [41]:
print(result["research_results"])

['### Historical milestones and timeline of GPU development over the last decade ###\nOver the past decade, GPU evolution has been driven by a relentless push for performance‑per‑watt, compute versatility, and real‑time rendering fidelity. In 2013 NVIDIA’s Pascal (GP104) launched the GeForce GTX\u202f10 series, establishing a new benchmark for power efficiency that AMD’s Vega followed in 2016 with HBM2 memory and deep‑learning‑optimized Volta. The 2018 Turing generation (RTX\u202f20) introduced dedicated RT and Tensor cores, enabling the first practical hardware ray‑tracing in games and the DLSS 1.0 up‑scaling technique; AMD’s RDNA (RX\u202f5000) responded with a GPU‑centric architecture for lower power consumption. 2019‑20 saw Ampere (RTX\u202f30) and RDNA2 (RX\u202f6000) double ray‑tracing throughput, while Intel debuted its Xe‑LP mobile GPUs, marking the first mainstream Intel discrete GPU. In 2021, NVIDIA’s Hopper (H100) shifted the focus to data‑center AI inference, and Intel’s Xe